# Step 1 and Step 2: checked Dice results

This notebook reads the committed results for the same 38 labeled patients held out from `Training` (seed 42). It does not train or rescore the models. The official `Validation` folder has no masks for Dice. See `EVALUATION.md` for reproduction steps and limitations.

In [ ]:
import csv
import json
from pathlib import Path

candidates = [Path.cwd(), *Path.cwd().parents]
project_root = next((candidate for base in candidates for candidate in (base, base / 'Abuya Code') if (candidate / 'outputs/evaluation/paired_patient_dice.csv').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Open the notebook from the UNIKL or Abuya Code project folder.')
results = project_root / 'outputs/evaluation'
with (results / 'paired_patient_dice.csv').open(newline='', encoding='utf-8') as file:
    paired = list(csv.DictReader(file))
with (results / 'onnx_step2/onnx_patient_dice.csv').open(newline='', encoding='utf-8') as file:
    onnx = list(csv.DictReader(file))
step1 = json.loads((results / 'paired_dice_summary.json').read_text(encoding='utf-8'))
step2 = json.loads((results / 'onnx_step2/onnx_dice_summary.json').read_text(encoding='utf-8'))
assert len(paired) == len(onnx) == 38
assert {row['case_id'] for row in paired} == {row['case_id'] for row in onnx}
print(f"Patients: {len(paired)}")
print(f"Zaq original: {step1['own_mean_dice']:.6f}")
print(f"Hayyi + radiomics: {step1['hayyi_mean_dice']:.6f}")
print(f"Zaq ONNX FP32: {step2['onnx_fp32_mean_dice']:.6f}")
print(f"Zaq ONNX INT8: {step2['onnx_int8_mean_dice']:.6f}")


## Interpretation

Zaq's original and INT8 means differ only slightly. Hayyi's radiomics features for these holdout patients were generated using ground-truth tumor masks, so the comparison with Hayyi is not a fully blind evaluation. The ONNX file is smaller; speed was not measured. Open `outputs/evaluation/step1_step2_comparison.html` for the patient-by-patient table and presentation notes.